# NixLMs 0.7 — Google Colab
Notebook oficial sincronizado com `main`. Inclui memória episódica, World Model, objetivos, atenção, recuperação semântica e integração opcional com Ollama. O núcleo funciona em CPU; Ollama e download de modelo são opcionais.

In [ ]:
from pathlib import Path
import subprocess, sys
REPO_DIR = Path('/content/NixLMs')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/SANTZSPAXX/NixLMs.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'], check=True)
%cd /content/NixLMs
# Descarta módulos importados anteriormente para evitar código Python em cache.
for module_name in list(sys.modules):
    if module_name == 'nixlms_core' or module_name.startswith('nixlms_core.'):
        del sys.modules[module_name]
globals().pop('ia', None)
from nixlms_core import NixLMsNeuralCore
import json, os
print('Python:', sys.version)
print('Checkout:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('Código importado de:', __import__('nixlms_core').__file__)

## Persistência entre sessões (Google Drive)
Execute esta célula para guardar memória, pesos, objetivos e dados no Drive. Se preferir armazenamento temporário da sessão, defina `USE_GOOGLE_DRIVE = False`.

In [ ]:
from pathlib import Path
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/NixLMs/data')
else:
    DATA_DIR = Path('/content/NixLMs/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
ia = NixLMsNeuralCore(data_dir=DATA_DIR)
if not ia.learning.dataset.load():
    print('Dataset inicial:', ia.learning.dataset.import_jsonl('datasets/starter_ptbr.jsonl'))
print(json.dumps(ia.status(), ensure_ascii=False, indent=2, default=str))

## Ollama local (opcional)
Execute somente se quiser geração de linguagem local. Esta célula instala Ollama e baixa `qwen2.5:0.5b`; isso consome internet, armazenamento e RAM/VRAM da sessão Colab. O modelo não é baixado automaticamente quando você executa o núcleo.

In [ ]:
import shutil, subprocess, time, urllib.request
from pathlib import Path
if shutil.which('ollama') is None:
    # O instalador oficial precisa de zstd para extrair o pacote no Colab.
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd'], check=True)
    subprocess.run(['bash', '-lc', 'curl -fsSL https://ollama.com/install.sh | sh'], check=True)
OLLAMA_BIN = shutil.which('ollama') or ('/usr/local/bin/ollama' if Path('/usr/local/bin/ollama').exists() else None)
if OLLAMA_BIN is None:
    raise RuntimeError('Instalação terminou sem localizar o binário ollama. Verifique a saída do instalador e o PATH.')
def ollama_ready():
    try:
        with urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2) as response:
            return response.status == 200
    except Exception:
        return False
ollama_log = open('/tmp/nixlms-ollama.log', 'a')
if not ollama_ready():
    ollama_process = subprocess.Popen([OLLAMA_BIN, 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
ready = False
for _ in range(30):
    if ollama_ready():
        ready = True
        break
    time.sleep(1)
if not ready:
    raise RuntimeError('Ollama não iniciou. Consulte /tmp/nixlms-ollama.log')
subprocess.run([OLLAMA_BIN, 'pull', 'qwen2.5:0.5b'], check=True)
print('Binário:', OLLAMA_BIN)
print('Modelos detectados:', ia.language.refresh_models())
print('Status:', ia.language.status())

## Conversa e comandos
Use `/llm` para verificar o serviço, `/llm usar NOME` para selecionar um modelo instalado, `/positivo` ou `/negativo` para avaliar a última resposta, `/ensinar pergunta => resposta` para registrar conhecimento e `/status` para inspecionar os módulos. O feedback afeta a recuperação episódica; não altera pesos neurais automaticamente. O split `validation` calibra o limiar LIF sem participar do treino; use `/calibrar-lif` depois de `/treinar`. `/avaliar` mede o fluxo integrado offline sem registrar episódios de teste; `/avaliar-llm` inclui o modelo Ollama opcional.

In [ ]:
def nix_chat():
    print('NixLMs 0.7 — digite /ajuda ou /sair')
    while True:
        message = input('Você: ').strip()
        low = message.lower()
        if low in {'/sair', '/exit', '/quit'}:
            print('Sessão encerrada.')
            break
        if not message:
            continue
        if low == '/ajuda':
            print('/status | /diagnostico | /llm | /llm usar NOME | /positivo | /negativo | /mapa | /avaliar | /treinar [epocas] | /ensinar pergunta => resposta | /sair')
        elif low == '/status':
            print(json.dumps(ia.status(), ensure_ascii=False, indent=2, default=str))
        elif low == '/diagnostico':
            print(json.dumps({'checkout': os.popen('git rev-parse --short HEAD').read().strip(), 'llm': ia.language.status(), 'last_trace': ia.last_trace}, ensure_ascii=False, indent=2, default=str))
        elif low == '/llm':
            available = ia.language.available()
            print('Ollama disponível:', available)
            print(ia.language.status())
        elif low.startswith('/llm usar '):
            try:
                print('Modelo selecionado:', ia.language.select_model(message[len('/llm usar '):].strip()))
            except ValueError as exc:
                print('Falha:', exc)
        elif low in {'/positivo', '/negativo'}:
            ok = ia.feedback(ia.last_question, low == '/positivo')
            print('Feedback registrado.' if ok else 'Não há episódio anterior correspondente.')
        elif low == '/mapa':
            print({'nodes': len(ia.mental_map.nodes), 'edges': len(ia.mental_map.edges), 'related': ia.mental_map.associate(ia.last_question, 10)})
        elif low.startswith('/importar '):
            try:
                imported = ia.learning.dataset.import_jsonl(message[len('/importar '):].strip())
                if imported:
                    ia.invalidate_lif_calibration()
                print('Registros importados:', imported)
            except (OSError, ValueError) as exc:
                print('Falha na importação:', exc)
        elif low == '/calibrar-decoder':
            try:
                print(ia.learning.calibrate_decoder_confidence())
            except ValueError as exc:
                print('Falha na calibração do decoder:', exc)
        elif low == '/calibrar-lif':
            print(ia.learning.calibrate_lif_threshold())
        elif low == '/avaliar-llm':
            print(ia.learning.evaluate(include_llm=True))
        elif low == '/avaliar':
            print(ia.learning.evaluate())
        elif low.startswith('/rollback '):\n            checkpoint_id = message.split(None, 1)[1].strip()\n            try:\n                print(ia.learning.restore_incremental_checkpoint(checkpoint_id))\n            except (OSError, ValueError) as exc:\n                print('Falha no rollback:', exc)\n        elif low.startswith('/treinar'):
            parts = low.split()
            print(ia.learning.train_incremental(int(parts[1]) if len(parts) > 1 else 1))
        elif low.startswith('/ensinar '):
            payload = message[len('/ensinar '):]
            if '=>' not in payload:
                print('Use /ensinar pergunta => resposta')
            else:
                question, answer = payload.split('=>', 1)
                ia.learn(question, answer, add_to_dataset=True)
                print('Conhecimento salvo.')
        else:
            print('NixLMs:', ia.respond(message, use_language=True))

nix_chat()

## Testes automatizados
A suíte usa mocks para serviços externos e não exige Ollama. A execução confirma o comportamento do código no runtime atual do Colab.

In [ ]:
!python -m unittest discover -s tests -v